# AdS4 Multi-Seed Branching Test (v14)

This notebook tests whether the phase transition identified in v10–v13 appears as **global branch selection** across different random initializations.

We:
- keep the same observable stack (EE + WL + GEO + consistency)
- train from many random seeds
- compare reconstructions below and above the empirical threshold c* ≈ 0.16
- visualize whether solutions collapse to one branch or split into multiple branches

Goal:
- make the global multi-solution structure directly visible


In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Ground-truth AdS4-style toy background
r = torch.linspace(1.05, 3.5, 240, device=device).view(-1, 1)

def f_true(r):
    return r**2 + 1 - 1/r

def g_true(r):
    return 1/(f_true(r) + 1e-6)

def ee_from_fg(f, g):
    return torch.log(1.0 + 0.6 * f)

def wl_from_fg(f, g):
    return torch.sqrt(f + 1.8 * g + 1e-6)

def geo_from_fg(f, g):
    return torch.sqrt(f + 0.7 * g + 1e-6)

ee_obs = ee_from_fg(f_true(r), g_true(r)).detach()
wl_obs = wl_from_fg(f_true(r), g_true(r)).detach()
geo_obs = geo_from_fg(f_true(r), g_true(r)).detach()


In [ ]:
def structured_geo_target(r, corruption_strength=0.0):
    base = geo_from_fg(f_true(r), g_true(r))
    if corruption_strength == 0.0:
        return base.detach()
    radial_bias = 1.0 + corruption_strength * (0.30 * (r - r.min()) / (r.max() - r.min()))
    oscillation = 1.0 + corruption_strength * 0.15 * torch.sin(2.2 * r)
    return (base * radial_bias * oscillation).detach()


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.f = MLP()
        self.g = MLP()
    def forward(self, r):
        f = F.softplus(self.f(r))
        g = F.softplus(self.g(r))
        return f, g


In [ ]:
def train_one(seed, corruption_strength=0.0, epochs=1200, consistency_weight=0.15):
    torch.manual_seed(seed)
    np.random.seed(seed)

    m = Model().to(device)
    opt = torch.optim.Adam(m.parameters(), lr=2e-3)
    geo_target = structured_geo_target(r, corruption_strength)

    loss_hist = []
    for _ in range(epochs):
        opt.zero_grad()
        f, g = m(r)

        ee_pred = ee_from_fg(f, g)
        wl_pred = wl_from_fg(f, g)
        geo_pred = geo_from_fg(f, g)

        loss_ee = ((ee_pred - ee_obs)**2).mean()
        loss_wl = ((wl_pred - wl_obs)**2).mean()
        loss_geo = ((geo_pred - geo_target)**2).mean()
        loss_consistency = ((f * g - 1.0)**2).mean()
        loss = loss_ee + loss_wl + loss_geo + consistency_weight * loss_consistency
        loss.backward()
        opt.step()
        loss_hist.append(loss.item())

    with torch.no_grad():
        f_pred, g_pred = m(r)
        metric_err = ((f_pred - f_true(r)).abs() + (g_pred - g_true(r)).abs()) / 2.0

    return {
        'seed': seed,
        'f_pred': f_pred.detach().cpu(),
        'g_pred': g_pred.detach().cpu(),
        'loss_hist': loss_hist,
        'mean_err': metric_err.mean().item(),
        'max_err': metric_err.max().item(),
    }


In [ ]:
# Run many seeds below and above threshold
seeds = list(range(8))
c_below = 0.00
c_above = 0.30

runs_below = [train_one(seed=s, corruption_strength=c_below) for s in seeds]
runs_above = [train_one(seed=s, corruption_strength=c_above) for s in seeds]

print('below errors:', [round(x['mean_err'], 4) for x in runs_below])
print('above errors:', [round(x['mean_err'], 4) for x in runs_above])


In [ ]:
# Measure spread across seeds as a proxy for branching
F_below = np.stack([x['f_pred'].squeeze().numpy() for x in runs_below], axis=0)
G_below = np.stack([x['g_pred'].squeeze().numpy() for x in runs_below], axis=0)
F_above = np.stack([x['f_pred'].squeeze().numpy() for x in runs_above], axis=0)
G_above = np.stack([x['g_pred'].squeeze().numpy() for x in runs_above], axis=0)

spread_f_below = F_below.std(axis=0)
spread_g_below = G_below.std(axis=0)
spread_f_above = F_above.std(axis=0)
spread_g_above = G_above.std(axis=0)

print('mean seed spread below:', float(spread_f_below.mean()), float(spread_g_below.mean()))
print('mean seed spread above:', float(spread_f_above.mean()), float(spread_g_above.mean()))


In [ ]:
# Main branching figure
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

for run in runs_below:
    axes[0].plot(r.cpu(), run['f_pred'], alpha=0.65)
axes[0].plot(r.cpu(), f_true(r).cpu(), color='black', linewidth=2, label='f true')
axes[0].set_title('below c*: f(r) across seeds')
axes[0].legend(fontsize=8)

for run in runs_below:
    axes[1].plot(r.cpu(), run['g_pred'], alpha=0.65)
axes[1].plot(r.cpu(), g_true(r).cpu(), color='black', linewidth=2, label='g true')
axes[1].set_title('below c*: g(r) across seeds')
axes[1].legend(fontsize=8)

axes[2].plot(r.cpu(), spread_f_below, label='f spread below')
axes[2].plot(r.cpu(), spread_g_below, label='g spread below')
axes[2].set_title('below c*: seed spread')
axes[2].legend(fontsize=8)

for run in runs_above:
    axes[3].plot(r.cpu(), run['f_pred'], alpha=0.65)
axes[3].plot(r.cpu(), f_true(r).cpu(), color='black', linewidth=2, label='f true')
axes[3].set_title('above c*: f(r) across seeds')
axes[3].legend(fontsize=8)

for run in runs_above:
    axes[4].plot(r.cpu(), run['g_pred'], alpha=0.65)
axes[4].plot(r.cpu(), g_true(r).cpu(), color='black', linewidth=2, label='g true')
axes[4].set_title('above c*: g(r) across seeds')
axes[4].legend(fontsize=8)

axes[5].plot(r.cpu(), spread_f_above, label='f spread above')
axes[5].plot(r.cpu(), spread_g_above, label='g spread above')
axes[5].set_title('above c*: seed spread')
axes[5].legend(fontsize=8)

plt.suptitle('AdS4 Multi-Seed Branching Test (v14)')
plt.tight_layout()
plt.savefig('/mnt/data/ads4_phase_lock_v14_branching.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: /mnt/data/ads4_phase_lock_v14_branching.png')


In [ ]:
# Loss comparison by seed
plt.figure(figsize=(10,4))
for run in runs_below:
    plt.plot(run['loss_hist'], alpha=0.5, color='tab:blue')
for run in runs_above:
    plt.plot(run['loss_hist'], alpha=0.5, color='tab:orange')
plt.title('Training trajectories: below c* (blue) vs above c* (orange)')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.tight_layout()
plt.savefig('/mnt/data/ads4_phase_lock_v14_losses.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: /mnt/data/ads4_phase_lock_v14_losses.png')
